# Looking for cloud frequency - is it random?

In [ ]:
import pystac
import planetary_computer
import rioxarray
import numpy as np
import geopandas as gpd
from pystac_client import Client
import xarray as xr

If you go via this link https://planetarycomputer.microsoft.com/explore?c=30.0586%2C29.9930&z=2.00&v=2 and 'explore' the code below is automatically created for you

In [ ]:
# As supplied by Planetary Computer


item_url = "https://planetarycomputer.microsoft.com/api/stac/v1/collections/sentinel-2-l2a/items/S2B_MSIL2A_20250405T110619_R137_T30UXB_20250405T132414"

# Load the individual item metadata and sign the assets
item = pystac.Item.from_file(item_url)

signed_item = planetary_computer.sign(item)

# Open one of the data assets (other asset keys to use: 'B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09', 'B11', 'B12', 'B8A', 'SCL', 'WVP', 'visual')
asset_href = signed_item.assets["SCL"].href
ds = rioxarray.open_rasterio(asset_href)
ds

Add a bounding box / Area of Interest

In [ ]:
gdf = gpd.read_file(r'...\map.geojson')

gdf

Query the STAC for Sentinel 2, a single scene intersecting my boundary (its a small boundary btw)

In [ ]:
stac_url = "https://planetarycomputer.microsoft.com/api/stac/v1"
client = Client.open(
    stac_url,
    modifier=planetary_computer.sign_inplace,
    timeout=300
)

# Sentinel-2 data
search = client.search(
    collections=["sentinel-2-l2a"],
    intersects=gdf.geometry[0], # ensure we just take the first layer
    datetime='2025-01-01/2025-05-01'
)

# Fetch items from the search results
items = list(search.items())

print(len(items))

I noticed on the STAC there was a parameter call no data pixel percentage and with a bit of googling... added the query to the above so I just return full scenes

In [ ]:
stac_url = "https://planetarycomputer.microsoft.com/api/stac/v1"
client = Client.open(
    stac_url,
    modifier=planetary_computer.sign_inplace,
    timeout=300
)

# Sentinel-2 data
search = client.search(
    collections=["sentinel-2-l2a"],
    intersects=gdf.geometry[0], # ensure we just take the first layer
    datetime='2025-01-01/2025-12-31',
    query={
        "s2:nodata_pixel_percentage": {
            "eq": 0
        }
    }
)

# Fetch items from the search results
items = list(search.items())

print(len(items))

Now I want to do something with the data - based on what planetary computer supplied, loading them into an xarray dataset made sense. Not sure if this is optimal (ie created a list and appending the datasets to it) - but worked.

Realised I need to merge on time to create the dataset I wanted - scl_combined

In [ ]:
# Load all SCL assets into a list of xarray DataArrays
scl_datasets = []
for item in items:
    signed_item = planetary_computer.sign(item)
    asset_href = signed_item.assets["SCL"].href
    # load into xarray
    ds = rioxarray.open_rasterio(asset_href, masked=True)
    scl_datasets.append(ds)


# merge on time
scl_combined = xr.concat(scl_datasets, dim="time")
scl_combined

Based on this blog https://medium.com/aimonks/understanding-sentinel-2-l2a-scene-classification-map-with-python-codes-5973938c95d4 I took cloud to be 8 or 9 category

and wrote out to a raster

In [ ]:
# SCL value is 8 or 9 (high or medium cloud)
cloud_mask = (scl_combined == 8) | (scl_combined == 9)

# Sum along the time dimension to count how many times each pixel was 'cloudy'
cloud_count = cloud_mask.sum(dim="time")

# percentage of cloudy observations
cloud_frequency = cloud_count / scl_combined.sizes["time"]

# write to file
cloud_frequency.rio.to_raster("cloud_frequency_2022.tif")